# Tutorial 1: The null model of "normal" progress

KAIROS fits Prophet to `ln y`. A piecewise-linear trend on the log scale is
piecewise **exponential** growth: the boring world. This notebook builds that
intuition on synthetic data with plain NumPy, so it runs anywhere. The last
cell repeats the fit with Prophet if it is installed.

The key fact: under exponential growth the growth rate `g = d ln x / dt` does
not depend on how big `x` is.

In [ ]:
import numpy as np

try:
    import matplotlib.pyplot as plt
except ImportError:  # plots are optional
    plt = None

rng = np.random.default_rng(13)

## A synthetic capability series

Monthly data, 2010 to 2026, with one regime change in 2017 (think transformers).

In [ ]:
t = np.arange(2010, 2026, 1 / 12)
slope = np.where(t < 2017, 0.6, 1.4)          # e-folds per year
ln_x = np.cumsum(slope) / 12 + rng.normal(0, 0.08, t.size)
print(f"{t.size} monthly points, ln x from {ln_x[0]:.2f} to {ln_x[-1]:.2f}")

## Piecewise-linear trend on `ln x`

A least-squares fit with hinge functions at candidate changepoints. This is what Prophet's trend does, minus the sparsity prior.

In [ ]:
knots = np.arange(2011, 2025, 1.0)
X = np.column_stack([np.ones_like(t), t - t[0]] + [np.clip(t - k, 0, None) for k in knots])
beta, *_ = np.linalg.lstsq(X, ln_x, rcond=None)
trend = X @ beta
rate_changes = dict(zip(knots, beta[2:]))
biggest = max(rate_changes, key=lambda k: abs(rate_changes[k]))
print(f"largest slope change at {biggest:.0f}: {rate_changes[biggest]:+.2f} e-folds/yr")

In [ ]:
g = np.gradient(trend, t)
print(f"growth rate before 2017: {g[t < 2016].mean():.2f}/yr, after: {g[t > 2018].mean():.2f}/yr")
if plt is not None:
    fig, ax = plt.subplots(1, 2, figsize=(10, 3.5))
    ax[0].plot(t, ln_x, ".", ms=3, label="ln x")
    ax[0].plot(t, trend, label="piecewise trend")
    ax[0].legend()
    ax[1].plot(t, g)
    ax[1].set_title("g = d ln x / dt (a step function)")
    plt.show()

## Why the null model is deliberately boring

The trend's growth rate is a **step function** (design note D1). It can never
produce `δ > 0` by itself, so KAIROS does *not* estimate acceleration from the
Prophet trend. It uses Prophet as the baseline that reality has to beat.

Repeated one-sided residuals against this baseline are the first hint that the
world is not boring.

In [ ]:
try:
    import pandas as pd
    from prophet import Prophet
except ImportError:
    print("prophet not installed; skipping the Prophet fit")
else:
    df = pd.DataFrame({"ds": pd.date_range("2010-01-01", periods=t.size, freq="MS"), "y": ln_x})
    m = Prophet(changepoint_prior_scale=0.3, interval_width=0.9, yearly_seasonality=False)
    m.fit(df)
    fc = m.predict(m.make_future_dataframe(periods=36, freq="MS"))
    print(fc[["ds", "yhat", "yhat_lower", "yhat_upper"]].tail(3))